In [ ]:
import numpy as np 
import pandas as pd 

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

count = 0
max_files = 10

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        count += 1
        if count >= max_files:
            break
    if count >= max_files:
        break

I am a B.Tech student targeting Fresher / Internship roles in Data Science (DS) and Machine Learning Engineering (MLE). I am looking to build a high-impact, standout portfolio project based on the WorldStrat / Multi-Frame Satellite Super-Resolution (MFSR) domain (upscaling 10m Sentinel-2 satellite revisits to high-resolution imagery).
also its a multiband thing not just rgb and we wil also provde a confidnece mask too in ui
I analyzed the original NeurIPS research repository (worldstrat) and noticed it is heavily over-engisneered for academic benchmarking (15+ scripts in dataset_generation for ordering Airbus imagery, 20+ modular classes, heavy PyTorch Lightning/WandB CLI wrappers).

Instead of blindly cloning that bloated research structure or trying to retrain on 100+ GB of data from scratch, my proposed approach is to build an applied, lean, end-to-end system: with training inferencing fronted,backined,basic mlops

Model & Training Pipeline:
Distill the core neural network (HighResNet / multi-frame fusion CNN) into a single, clean PyTorch script/Jupyter Notebook without unnecessary abstract classes.
Use a small, curated subset of satellite AOIs (50–100 locations) to train/fine-tune the model on Google Colab (T4 GPU) using a standard loss (L1 + SSIM) and log clean loss curves.
Backend & Inference API:
Build a lightweight FastAPI microservice serving the trained checkpoint with endpoints for inference, preprocessing, and metric evaluation (comparing model output vs. a naive bicubic baseline).
Frontend & Visualization:
Develop a modern interactive web UI with a split-screen 'Before vs. After' slider comparing low-res multi-pass input to the super-resolved output, plus multi-spectral band visualization.
MLOps & Engineering Best Practices:
Containerize the app using Docker.
Add basic CI/CD via GitHub Actions (linting, automated pytest unit tests for tensor shapes and API responses).
Deploy on a free cloud tier (e.g., Hugging Face Spaces / Render).
My Questions for You:

Is this a strong and realistic project for a B.Tech fresher/intern resume targeting MLE vs. DS roles? Does it stand out compared to typical CV projects?
Is this distillation approach (single clean training notebook or a good backind architure  + lean FastAPI service) the right way to handle research code, or are there hidden pitfalls? only having few paramters not too many
What specific metrics, optimizations, or architectural decisions should I be prepared to defend during technical interviews?
How would you rate the scope, complexity, and resume impact of this project on a scale of 1–10?" 5.also then i belive i should diverty away from this worldstrat code (in terms of directly making mines projct on worldstrat code modifid  snipps i belive i should make minesa anbd use worldtraat as reference ) too as reverseeinginner it is way to complex or maybe i might be donig wrong as nonthelss this also improves my skill to read others code and debug it too you tell my mind behind this was as its has wayyy  wayy too much parameters which alone will take a good time for me to undestand in this modular code approch that it might defate the purpose of a projct you tell mw what to do analuyse this whol projct everyt file and its paramters before commeing 6. and shuld i retrain on a small dataset or should i do tranfer leaning my fear is this i won't be able to show my model architreucre skill if i do not haveign a trainig part 

All 22 Micah's locations with an AOI area of 22.5 km², with a total area of 495 km².
All 40 of ASMSpotter locations with an AOI area of 22.5 km², with a total area of 900 km².
Randomly selected 1000 UNHCR POIs, with an AOI area of 2.5 km² and a total area of 2500 km².
2442 randomly sampled points, with an AOI area of 2.5 km² and a total area of 6105 km².

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/jucor1/worldstrat/metadata.csv",index_col=0)
df.drop("n",axis = 1,inplace = True)
mask = ~df.index.duplicated(keep='first')
df = df[mask]
df.head()

In [ ]:
df.reset_index(inplace=True)

In [ ]:
df['bounds'] = df['bounds'].apply(eval)

In [ ]:
df

In [ ]:
df.info(),df.shape,df.dtypes,df.columns

In [ ]:
df.describe

In [ ]:
def bounds_to_bounding_box(
    lat_min, lon_min, lat_max, lon_max, closed=False, lonlat=False
):
    """Convert bounds to a bounding box (square).

    Parameters
    ----------
    lat_min : float
        Minimum latitude boundary.
    lon_min : float
        Minimum longitude boundary.
    lat_max : float
        Maximum latitude boundary.
    lon_max : float
        Maximum longitude boundary.
    closed : bool, optional
        Closes back to the first corner if True, by default False.
    lonlat : bool, optional
        Points in the bb are in the [longitude, latitude] order if True, in [latitude, longitude] if False. Default False.

    Returns
    -------
    list
        Four corners defining a bounding box, goes clockwise from SouthWest.
    """
    if lonlat:
        bounding_box = [
            [lon_min, lat_min],
            [lon_min, lat_max],
            [lon_max, lat_max],
            [lon_max, lat_min],
        ]
        if closed:
            bounding_box.append([lon_min, lat_min])
    else:
        bounding_box = [
            [lat_min, lon_min],
            [lat_min, lon_max],
            [lat_max, lon_max],
            [lat_max, lon_min],
        ]

        if closed:
            bounding_box.append([lat_min, lon_min])

    return bounding_box

In [ ]:
from folium.plugins import MarkerCluster
import folium
from shapely.geometry.polygon import Polygon
import plotnine as p9
from plotnine import ggplot, geom_bar, aes, ggtitle, xlab, ylab, facet_wrap

#from dataset_generation.AOIGenerator import AOIGenerator
import tifffile as tiff
from PIL import Image
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
import imageio
from IPython.display import Image as IPythonImage
import os
from tqdm.notebook import tqdm
import pandas as pd

class Visualiser:
    """ Provides visualisations for all classes. """

    def __init__(self, data):
        """ Initialises the visualiser with the data to be visualised. 

        Parameters
        ----------
        data : pandas.DataFrame
            The data to be visualised.
        """
        self.data = data

    def update_data(self, data):
        """ Updates the data to be visualised.

        Parameters
        ----------
        data : pandas.DataFrame
            The data to be visualised.
        """
        self.data = data

    def visualise_points_on_map(self):
        """ Visualises points on a world map.

        Returns
        -------
        folium.Map
            The world map with the points visualised.
        """
        map = folium.Map([0, 0], zoom_start=3, tiles="CartoDB dark_matter")
        folium.TileLayer(
            "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community",
        ).add_to(map)

        marker_cluster = MarkerCluster().add_to(map)
        for _, point in tqdm(self.data.iterrows(), total=len(self.data), desc="Visualising points on a map"):
            folium.Marker((point["lat"], point["lon"])).add_to(marker_cluster)
        folium.LatLngPopup().add_to(map)
        return map

    def visualise_aois_on_map(self):
        """ Visualises AOIs on a world map.

        Returns
        -------
        folium.Map
            The world map with the AOIs visualised.
        """
        map = folium.Map([0, 0], zoom_start=3, tiles="CartoDB dark_matter")
        folium.TileLayer(
            "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
            attr="Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community",
        ).add_to(map)
        aoi_style = {"fillColor": "#e74c3c", "color": "#c0392b"}
        marker_cluster = MarkerCluster().add_to(map)
        use_tooltip = pd.Series(['IPCC', 'LCCS', 'SMOD']).isin(self.data.columns).all() 
        for _, aoi in tqdm(self.data.iterrows(), total=len(self.data), desc='Visualising AOIs on map'):
            folium.Marker((aoi["lat"], aoi["lon"])).add_to(marker_cluster)
            aoi_polygon = Polygon(bounds_to_bounding_box(*aoi["bounds"]))
            aoi_geojson = folium.GeoJson(
                aoi_polygon, style_function=lambda x: aoi_style
            )
            if use_tooltip:
                aoi_tooltip = folium.Tooltip(
                    f"<strong>AOI:</strong> {aoi['name']} <br> <strong>IPCC:</strong> {aoi['IPCC']} <br> <strong>LCCS:</strong> {aoi['LCCS']} <br> <strong>SMOD:</strong> {aoi['SMOD']}"
                )
                aoi_tooltip.add_to(aoi_geojson)
            aoi_geojson.add_to(map)
        folium.LatLngPopup().add_to(map)
        return map

  

In [ ]:
sample_df = df.sample(n=50, random_state=42)
visualiser = Visualiser(sample_df)
visualiser.update_data(sample_df)
visualiser.visualise_aois_on_map()

****1.Dataset Exploration> ****

In [ ]:
path = "/kaggle/input/datasets/jucor1/worldstrat/lr_dataset/ASMSpotter-1-1-2/L2A/ASMSpotter-1-1-2-1-L2A_data.tiff"
import rasterio
from rasterio.plot import show
import numpy as np
"""L2A data through the SentinelHub API, which already converted the Bottom-of-Atmosphere (BOA) surface reflectance directly into float32 physical decimals:

0.0 = 0% reflectance
0.15 = 15% reflectance (vegetation/soil)
0.30 = 30% reflectance (the cutoff for normal ground features!)
1.0 = 100% reflectance (clouds/snow)


so no need of normalization here
"""
# 1. Open with rasterio
with rasterio.open(path) as src:
    # Read Bands 4 (Red), 3 (Green), 2 (Blue) -> returns shape: (3, H, W)
    rgb_data = src.read([4, 3, 2])
    print("Shape:", rgb_data.shape)
    print("Dtype:", rgb_data.dtype)
    print(f"Min: {rgb_data.min()}, Max: {rgb_data.max()}, Mean: {rgb_data.mean():.4f}, Median: {np.median(rgb_data):4f}")
    show(src.read(2), transform=src.transform, cmap='viridis')
    show(rgb_data, transform=src.transform,adjust=None, title="Sentinel-2 True Color (Rasterio)")

channel_means = rgb_data.mean(axis=(1, 2))

In [ ]:
#per  channel mean
channel_means[0],channel_means[1],channel_means[2]

In [ ]:
from matplotlib import  pyplot
with rasterio.open(path) as src:
    fig, (axr, axg, axb) = pyplot.subplots(1,3, figsize=(21,7))
    show((src, 1), ax=axr, cmap='Reds', title='red channel')
    show((src, 2), ax=axg, cmap='Greens', title='green channel')
    show((src, 3), ax=axb, cmap='Blues', title='blue channel')
    pyplot.show()

In [ ]:
from rasterio.plot import show_hist
src = rasterio.open(path)
fig, (axrgb, axhist) = pyplot.subplots(1, 2, figsize=(14,7))
show(src, ax=axrgb)
show_hist(src, bins=50, histtype='stepfilled',
          lw=0.0, stacked=False, alpha=0.3, ax=axhist)
pyplot.show()

In [ ]:
from collections import Counter
from pathlib import Path

# Set the path to your directory
target_path = Path("/kaggle/input/datasets/jucor1/worldstrat/lr_dataset")

folder_count = 0
file_formats = Counter()

# Loop through the top-level items
for item in target_path.iterdir():
    if item.is_dir():
        folder_count += 1
    elif item.is_file():
        # Get extension in lowercase (e.g., '.txt', '.png')
        # If a file has no extension, it uses 'No Extension'
        ext = item.suffix.lower() if item.suffix else "no extension"
        file_formats[ext] += 1

# --- Print Results ---
print(f"Total Folders: {folder_count}")
print(f"Total Files:   {sum(file_formats.values())}\n")

print("File Formats Break Down:")
for ext, count in file_formats.items():
    print(f"  * {ext}: {count} file(s)")


In [ ]:
path = "/kaggle/input/datasets/jucor1/worldstrat/hr_dataset/12bit/ASMSpotter-1-1-2/ASMSpotter-1-1-2_ps.tiff"
#At this stage, src is a DatasetReader object. It only holds file properties
with rasterio.open(path) as src:
    data = src.read()#Loads pixels into RAM: This reads the actual pixel data off disk into memory.
    print("Shape:", data.shape)  
    print("Dtype:", data.dtype)  
    print(f"Min: {data.min()}")
    print(f"Max: {data.max()}")
    print(f"Mean: {data.mean():.4f}")
    print(f"Median: {np.median(data):.4f}")
    show(data, transform=src.transform)

In [ ]:
import numpy as np
import rasterio

path = "/kaggle/input/datasets/jucor1/worldstrat/hr_dataset/12bit/ASMSpotter-1-1-2/ASMSpotter-1-1-2_ps.tiff"

with rasterio.open(path) as src:
    # 1. Storage container data type per band (e.g., 'uint8', 'uint16')
    dtypes = src.dtypes
    
    # 2. Check if the GeoTIFF explicitly tags bit depth in metadata
    explicit_bits = [src.tags(b).get("NBITS", "Not explicitly tagged") for b in range(1, src.count + 1)]
    
    # 3. Read pixel data to inspect the observed bit depth from actual values
    data = src.read()
    max_val = data.max()
    
    # Calculate bits needed to represent the maximum observed value: ceil(log2(max + 1))
    observed_bits = int(np.ceil(np.log2(max_val + 1))) if max_val > 0 else 0

    print(f"Number of bands: {src.count}")
    print(f"Data type per band: {dtypes}")
    print(f"Metadata NBITS tag: {explicit_bits}")
    print(f"Max pixel value: {max_val}")
    print(f"Effective bit depth (from values): ~{observed_bits}-bit")

In [ ]:
import numpy as np
import rasterio

path = "/kaggle/input/datasets/jucor1/worldstrat/lr_dataset/ASMSpotter-1-1-2/L2A/ASMSpotter-1-1-2-1-L2A_data.tiff"

with rasterio.open(path) as src:
    # 1. Storage container data type per band (e.g., 'uint8', 'uint16')
    dtypes = src.dtypes
    
    # 2. Check if the GeoTIFF explicitly tags bit depth in metadata
    explicit_bits = [src.tags(b).get("NBITS", "Not explicitly tagged") for b in range(1, src.count + 1)]
    
    # 3. Read pixel data to inspect the observed bit depth from actual values
    data = src.read()
    max_val = data.max()
    
    # Calculate bits needed to represent the maximum observed value: ceil(log2(max + 1))
    observed_bits = int(np.ceil(np.log2(max_val + 1))) if max_val > 0 else 0

    print(f"Number of bands: {src.count}\n")
    print(f"Data type per band: {dtypes}\n")
    print(f"Metadata NBITS tag: {explicit_bits}\n")
    print(f"Max pixel value: {max_val}\n")
    print(f"Effective bit depth (from values): ~{observed_bits}-bit")

In [ ]:
#show_aoi = interact_manual.options(manual_name="Show AOI")

In [ ]:
dataset_root = '/kaggle/input/datasets/jucor1/worldstrat'
hr_dataset_folder = 'hr_dataset/'
lr_dataset_folder = 'lr_dataset/'
import os
root=os.path.join(dataset_root, lr_dataset_folder, "*", "L2A", "")
print(root)

In [ ]:
import os
import glob
# 1. Build the path string with the wildcard
root_pattern = os.path.join(dataset_root, lr_dataset_folder, "*", "L2A", "")
root_pattern2=os.path.join(dataset_root, hr_dataset_folder,"12bit", "*")
# 2. Search for all paths that match the pattern
actual_paths = glob.glob(root_pattern)
actual_paths2 = glob.glob(root_pattern2)

print(f"Found {len(actual_paths)} matching directories.")
print(f"Found {len(actual_paths2)} matching directories.")

In [ ]:
from glob import glob
print(glob("/kaggle/input/datasets/jucor1/worldstrat/lr_dataset/*/L2A/")[:5])

In [ ]:
import os
from glob import glob
from multiprocessing import Manager
import natsort
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset
from torchvision.transforms import Compose, Resize, InterpolationMode, Normalize, Lambda

#i din't include compyut_median_std() as i pasted precomputed values and also panchromatic conversoin tooo as we are working 
# on pre computed dataset
class SatelliteDataset(Dataset):
    """PyTorch Dataset class used to load satellite imagery. """

    def __init__(
        self,
        root,
        number_of_revisits=8,
        subdir="",
        file_postfix=None,
        transform=None,
        bands_to_read=None,
        file_reader=None,
        use_cache=True,
        multiprocessing_manager=None,
    ):
        """PyTorch Dataset class used to load satellite imagery.

        Parameters
        ----------
        root : str
            Path to a directory that contains either one subdirectory per AOI or one file per AOI revisit.
            Every subdirectory would contain the revisits for that AOI.

        subdir : str, optional
            Subdirectory within the root where revisits are, by default '' (no subdirectory).

        file_postfix : str, optional
            The postfix in which the revisit files end, by default 'tif*'.
            Allows wildcards.

        number_of_revisits : int, optional
            The number of revisits to get for each AOI, by default None (all revisits).

        transform : torchvision.transform, optional
            A transform that is applied to each revisit, by default Compose([]) (no transform).

        bands_to_read : list, optional
            A list of band IDs to read, by default None.
            By GDAL convention, band IDs are 1-indexed.

        file_reader : callable, optional
            A callable function used to read the files, by default SatelliteDataset.read_tiff_with_rasterio.

        use_cache : bool, optional
            Cache fetched items to memory, by default True.
            Can require a lot of RAM, but greatly speeds up training.

        multiprocessing_manager : multiprocessing.Manager, optional
            Used to prevent CPU memory leaking when multiple DataLoader workers access the cache, by default None.
            multiprocessing.Manager data structures are used to prevent copy-on-write behaviour due to refcounts in Python.
            See: https://github.com/pytorch/pytorch/issues/13246.
        """

        self.check_root_is_not_empty(root)
        self.root = root
        self.subdir = subdir
        self.file_postfix = file_postfix if file_postfix is not None else "tif*"
        self.file_reader = file_reader or self.read_tiff_with_rasterio
        self.transform = transform or Compose([])
        self.bands_to_read = list(bands_to_read) if bands_to_read is not None else None
        self.number_of_revisits = number_of_revisits
        self.multiprocessing_manager = multiprocessing_manager
        self.paths = self.load_and_sort_aoi_paths(root)
        self.use_cache = use_cache
        
        if self.use_cache:
            if self.multiprocessing_manager is None:
                self.manager = Manager()
                self.cache = self.manager.dict()
            else:
                self.cache = self.multiprocessing_manager.dict()
        else:
            self.cache = None

    @staticmethod
    def check_root_is_not_empty(root):
        """Checks that the root is not empty.

        Parameters
        ----------
        root : str
            Path to a directory that contains either one subdirectory per AOI or one file per AOI revisit.
        """
        assert len(glob(root)) > 0, f"No files from {root}."

    def __len__(self):
        """Returns the number of AOIs in the dataset.

        Returns
        -------
        int
            The number of AOIs in the dataset.
        """
        return len(self.paths)

    def __getitem__(self, item: int):
        """Returns a satellite image for a given AOI index from the dataset.
        If the image was previously fetched, it is returned from the cache.
        Otherwise, it is fetched from the disk and transformed.

        Parameters
        ----------
        item : int
            The index of the AOI in the dataset.

        Returns
        -------
        Tensor
            The satellite image as a tensor.
        """
        if not self.use_cache or item not in self.cache:
            path = self.paths[item]
            if os.path.isdir(path):
                # The path is a folder
                return self.load_revisits_from_folder(path, item)
            else:
                # The path is a single file
                x = self.file_reader(path, self.bands_to_read)
                x = self.transform(x)
                if self.use_cache:
                    self.cache[item] = x
                return x
        else:
            # The item is cached, return from cache
            return self.cache[item]

    @staticmethod
    def read_tiff_with_rasterio(path, bands_to_read=None):
        """Reads a TIFF file using rasterio.

        Parameters
        ----------
        path : str
            The path to the TIFF file.
        bands_to_read : list, optional
            A list of band IDs to read, by default None.

        Returns
        -------
        Tensor
            The image as a tensor.
        """
        data = rasterio.open(path).read(indexes=bands_to_read, window=None)
        return torch.as_tensor(data.astype(np.float32))

    def load_and_sort_aoi_paths(self, root):
        """Loads the AOI paths from the root and sorts them.

        Parameters
        ----------
        root : str
            The path to the root directory.

        Returns
        -------
        multiprocessing.Manager.list
            A list of AOI paths, naturally sorted.
        """
        if self.multiprocessing_manager is not None:
            return self.multiprocessing_manager.list(natsort.natsorted(glob(root)))
        else:
            return natsort.natsorted(glob(root))

    def load_revisits_from_folder(self, path, item):
        """Loads the revisits/satellite images for a given AOI from a folder.

        Parameters
        ----------
        path : str
            The path to the AOI folder.
        item : int
            The index of the AOI in the dataset.

        Returns
        -------
        Tensor
            The revisits as a tensor.
        """
        # The folder contains revisits for a single AOI, read them into an AOI-specific dataset
        search_path = os.path.join(path, self.subdir, f"*{self.file_postfix}")
        revisit_files = natsort.natsorted(glob(search_path))

        if len(revisit_files) == 0:
            return None

        # Determine the number of revisits to return
        if self.number_of_revisits:
            if len(revisit_files) < self.number_of_revisits:
                revisit_files = (revisit_files * ((self.number_of_revisits // len(revisit_files)) + 1))[:self.number_of_revisits]
            else:
                revisit_files = revisit_files[:self.number_of_revisits]

        # Stack along revisit/time dimension -> (revisits, bands, height, width)
        revisits = [self.file_reader(f, self.bands_to_read) for f in revisit_files]
        x = torch.stack(revisits, dim=0)

        x = self.transform(x)
        self.cache_revisits(item, x)
        return x

    def cache_revisits(self, item, x):
        """Caches the AOI revisits within the multiprocessing.Manager dictionary.

        Parameters
        ----------
        item : int
            The index of the AOI in the dataset.
        x : Tensor
            The revisits as a tensor.
        """
        if self.use_cache:
            self.cache[item] = x


In [ ]:

class DictDataset(Dataset):
    """ PyTorch Dataset class used to wrap a dictionary of datasets.
    The item index passed to the get method refers to images from a single AOI, or patch (when cropping is applied).
    The get method returns a dictionary of the items at the item index for each dataset."""

    def __init__(self, **dictionary_of_datasets):
        """

        Parameters
        ----------
        dictionary_of_datasets : dict
            The dictionary of torch datasets to wrap.

                                                                    self.datasets = {
                                                                "lr": <SatelliteDataset instance for Sentinel-2>,
                                                                "hr": <SatelliteDataset instance for SPOT HR>
                                                            }

        """
        # self.datasets = {
        #     dataset_name: dataset
        #     for dataset_name, dataset in dictionary_of_datasets.items()
        #     if isinstance(dataset, Dataset)
        # }
        self.datasets = {}  # 1. Start with an empty dictionary
    
        for dataset_name, dataset in dictionary_of_datasets.items():  # 2. Loop through every pair
            if isinstance(dataset, Dataset):  # 3. Check if it's a PyTorch Dataset
                self.datasets[dataset_name] = dataset  # 4. If yes, save it in the dictionary


    def __getitem__(self, item):
        """Get an item from the datasets.

        Parameters
        ----------
        item : int
            The index of the item to get from each dataset.

        Returns
        -------
        dict
            The items from each dataset (dataset_name:item).
        """
        return {name: dataset[item] for name, dataset in self.datasets.items()}

    def __len__(self):
        """Returns the length of the smallest dataset in the dictionary of wrapped datasets.

        Returns
        -------
        int
            The length of the smallest dataset in the dictionary of wrapped datasets.
        """
        return min(len(dictionary) for dictionary in self.datasets.values())


In [ ]:
import os
import torch
from glob import glob

# ------------------------------------------------------ #
#                  Sentinel2   - S2
#                  Level 2A   - L2A
#                  Level 1C   - L1C
# ------------------------------------------------------ #


# Source: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/composites/
# NOTE: these are 1-indexed because GDAL is 1-indexed.
S2_ALL_12BANDS = {
    "true_color": [4, 3, 2],
    "false_color": [8, 4, 3],
    "swir": [12, 8, 4],
    "agriculture": [11, 8, 2],
    "geology": [12, 11, 2],
    "bathimetric": [4, 3, 1],
    "true_color_zero_index": [3, 2, 1],
}

S2_ALL_BANDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# ------------------------------------------------------ #
#                  SpaceNet7   - SN7
#                  PlanetScope - PS
# ------------------------------------------------------ #

PS_BANDS = [1, 2, 3]

METADATA_PATH = 'dataset/metadata.csv'
ROOT_SN7_DATA = os.path.join("data", "SN7")
ROOT_SN7_DATA_TRAIN = os.path.join(ROOT_SN7_DATA, "train")
SCENES_SN7 = glob(os.path.join(ROOT_SN7_DATA_TRAIN, "L15*"))

SN7_SUBDIRECTORIES = {"lr": "S2", "lrc": "S2_masks", "hr": "images"}
SN7_BANDS_TO_READ = {"lr": S2_ALL_12BANDS["true_color"], "lrc": None, "hr": PS_BANDS}


# Precomputed on training data by running: src/datasets.py
S2_SN7_MEAN = torch.Tensor(
    [
        0.0769,
        0.0906,
        0.1162,
        0.1318,
        0.1619,
        0.2275,
        0.2500,
        0.2588,
        0.2667,
        0.2684,
        0.2461,
        0.1874,
    ]
).to(torch.float64)

S2_SN7_STD = torch.Tensor(
    [
        0.1708,
        0.1672,
        0.1614,
        0.1677,
        0.1698,
        0.1513,
        0.1490,
        0.1515,
        0.1461,
        0.2365,
        0.1472,
        0.1418,
    ]
).to(torch.float64)

PS_MEAN = torch.Tensor([105.0, 92.0, 66.0]).to(torch.float64)  # RGB
PS_STD = torch.Tensor([58.2366, 47.1755, 44.9830]).to(torch.float64)
SN7_MAX_EXPECTED_HR_VALUE = 255


# ------------------------------------------------------ #
#   Satellite pour l'Observation de la Terre - SPOT 7
#   Julien, Ivan, Freddie (working name)     - JIF
# ------------------------------------------------------ #

SPOT_BANDS = [0, 1, 2, 3]
SPOT_RGB_BANDS = [1, 2, 3]
SPOT_MAX_EXPECTED_VALUE_8_BIT = 255
SPOT_MAX_EXPECTED_VALUE_12_BIT = 10000

ROOT_JIF_DATA = os.path.join("data", "JIF")
ROOT_JIF_DATA_TRAIN = os.path.join(ROOT_JIF_DATA, "train")

# Precomputed on training data by running: src/datasets.py
JIF_S2_MEAN = torch.Tensor(
    [
        0.0008,
        0.0117,
        0.0211,
        0.0198,
        0.0671,
        0.1274,
        0.1449,
        0.1841,
        0.1738,
        0.1803,
        0.0566,
        -0.0559,
    ]
).to(torch.float64)

JIF_S2_STD = torch.Tensor(
    [
        0.0892,
        0.0976,
        0.1001,
        0.1169,
        0.1154,
        0.1066,
        0.1097,
        0.1151,
        0.1122,
        0.1176,
        0.1298,
        0.1176,
    ]
).to(torch.float64)

URBAN_SPOT_MEAN = torch.Tensor([0.0025, 0.0022, 0.0017]).to(torch.float64)
URBAN_SPOT_STD = torch.Tensor([0.0021, 0.0018, 0.0017]).to(torch.float64)


In [ ]:
def make_transforms_JIF(
    input_size=(160, 160),
    output_size=(1054, 1054),
    interpolation=InterpolationMode.BICUBIC,
    normalize_lr=True,
    scene_classification_to_color=False,
    radiometry_depth=12,
    lr_bands_to_use="all",
    **kws,
):
    """ Make the transforms for the JIF dataset.
    The transforms normalize and resize the images to the appropriate sizes.
    """
    if radiometry_depth == 8:
        maximum_expected_hr_value = SPOT_MAX_EXPECTED_VALUE_8_BIT
    else:
        maximum_expected_hr_value = SPOT_MAX_EXPECTED_VALUE_12_BIT

    transforms = {}
    if lr_bands_to_use == "true_color":
        lr_bands_to_use = np.array(S2_ALL_12BANDS["true_color"]) - 1
    else:
        lr_bands_to_use = np.array(S2_ALL_BANDS) - 1

    if normalize_lr:
        normalize = Normalize(
            mean=JIF_S2_MEAN[lr_bands_to_use].float(), std=JIF_S2_STD[lr_bands_to_use].float()
        )
    else:
        normalize = Compose([])

    # Safe float32 tensor converter (works seamlessly on both torch.Tensor and np.ndarray)
    to_float_tensor = lambda img: img.float() if isinstance(img, torch.Tensor) else torch.as_tensor(img, dtype=torch.float32)

    transforms["lr"] = Compose(
        [
            Lambda(to_float_tensor),
            normalize,
            Resize(size=input_size, interpolation=interpolation, antialias=True),
        ]
    )

    transforms["lrc"] = Compose(
        [
            Lambda(to_float_tensor),
            Resize(size=input_size, interpolation=InterpolationMode.NEAREST),
            SceneClassificationToColorTransform
            if scene_classification_to_color
            else Compose([]),
        ]
    )

    transforms["hr"] = Compose(
        [
            Lambda(
                lambda hr_revisit: (
                    hr_revisit.float()
                    if isinstance(hr_revisit, torch.Tensor)
                    else torch.as_tensor(hr_revisit, dtype=torch.float32)
                )
                / maximum_expected_hr_value
            ),
            Resize(size=output_size, interpolation=interpolation, antialias=True),
            Lambda(lambda high_res_revisit: high_res_revisit.clamp(min=0, max=1)),
        ]
    )

    transforms["hr_pan"] = Compose(
        [
            Lambda(
                lambda hr_panchromatic: (
                    hr_panchromatic.float()
                    if isinstance(hr_panchromatic, torch.Tensor)
                    else torch.as_tensor(hr_panchromatic, dtype=torch.float32)
                )
                / maximum_expected_hr_value
            ),
            Resize(size=output_size, interpolation=interpolation, antialias=True),
            Lambda(
                lambda high_res_panchromatic: high_res_panchromatic.clamp(min=0, max=1)
            ),
        ]
    )
    return transforms


In [ ]:
import os

from ipywidgets import interact_manual, Layout
import ipywidgets as widgets
from glob import glob
from torch.utils.data import DataLoader
from torchvision.utils import make_grid


from multiprocessing import Manager
transforms = make_transforms_JIF(lr_bands_to_use='true_color', radiometry_depth=12)
multiprocessing_manager = Manager()

dataset_root = '/kaggle/input/datasets/jucor1/worldstrat/'
hr_dataset_folder = 'hr_dataset/12bit/'
lr_dataset_folder = 'lr_dataset/'

In [ ]:
lr = SatelliteDataset(
    root=os.path.join(dataset_root, lr_dataset_folder, "*", "L2A", ""),
    file_postfix="-L2A_data.tiff",
    transform=transforms["lr"],
    number_of_revisits=8,
    bands_to_read=S2_ALL_12BANDS["true_color"],
    multiprocessing_manager=multiprocessing_manager
)

lrc = SatelliteDataset(
    root=os.path.join(dataset_root, lr_dataset_folder, "*", "L2A", ""),
    file_postfix="-CLM.tiff",
    transform=transforms["lrc"],
    number_of_revisits=8,
    multiprocessing_manager=multiprocessing_manager
)

hr_panchromatic = SatelliteDataset(
    root=os.path.join(dataset_root, hr_dataset_folder, "*"),
    file_postfix="_pan.tiff",
    transform=transforms["hr_pan"],
    number_of_revisits=1,
    multiprocessing_manager=multiprocessing_manager
)

hr_rgb = SatelliteDataset(
    root=os.path.join(dataset_root, hr_dataset_folder, "*"),
    file_postfix="_rgbn.tiff",
    transform=transforms["hr"],
    bands_to_read=SPOT_RGB_BANDS,
    number_of_revisits=1,
    multiprocessing_manager=multiprocessing_manager
)

hr_pansharpened = SatelliteDataset(
    root=os.path.join(dataset_root, hr_dataset_folder, "*"),
    file_postfix="_ps.tiff",
    transform=transforms["hr"],
    bands_to_read=SPOT_RGB_BANDS,
    number_of_revisits=1,
    multiprocessing_manager=multiprocessing_manager
)

dataset = DictDataset(
    **{
        "lr": lr,
        "lrc": lrc,
        "hr": hr_rgb,
        "hr_pan": hr_panchromatic,
        "hr_pansharpened": hr_pansharpened,
    }
)

print(f"Loaded dataset with {len(dataset)} images.")

In [ ]:
# Cell 35: Enhanced Interactive Multi-Modal Image Viewer
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid
from ipywidgets import interact_manual, Layout
import ipywidgets as widgets

try:
    from src.plot import showtensor
except ImportError:
    def showtensor(tensor, figsize=10, title=None):
        if tensor.ndim == 4:
            tensor = tensor[0]
        img = tensor.permute(1, 2, 0).detach().cpu().numpy()
        plt.figure(figsize=(figsize, figsize // 2 if tensor.shape[1] > tensor.shape[2] * 2 else figsize))
        if title:
            plt.title(title, fontsize=12, pad=10)
        plt.imshow(np.clip(img, 0, 1))
        plt.axis('off')
        plt.show()

show_aoi = interact_manual.options(manual_name="Show AOI Imagery")

def view_image(i=0):
    item = dataset[i]
    lr, lrc, hr_rgbn, hr_pan, ps = item['lr'], item['lrc'], item['hr'], item['hr_pan'], item['hr_pansharpened']
    
    # Extract AOI name if available from paths or dataframe
    aoi_name = 'Unknown'
    if hasattr(dataset.datasets['lr'], 'paths') and i < len(dataset.datasets['lr'].paths):
        from pathlib import Path
        aoi_name = Path(dataset.datasets['lr'].paths[i]).name
    elif 'df' in globals() and 'index' in df.columns and i < len(df):
        aoi_name = df.iloc[i]['index']
    
    print(f"=== AOI [{i} / {len(dataset)-1}]: {aoi_name} ===")
    print(f"LR Shape: {tuple(lr.shape)} | HR Shape: {tuple(hr_rgbn.shape)} | Revisits: {len(lr)}")
    
    # 1. Standardize 4D (1, C, H, W) to 3D (C, H, W)
    rgb_img = hr_rgbn[0] if hr_rgbn.ndim == 4 else hr_rgbn
    pan_img = hr_pan[0]  if hr_pan.ndim == 4  else hr_pan
    ps_img  = ps[0]      if ps.ndim == 4      else ps
    
    # 2. Expand single-channel panchromatic (1, H, W) -> (3, H, W) for side-by-side grid
    if pan_img.shape[0] == 1:
        pan_3ch = pan_img.repeat(3, 1, 1)
    else:
        pan_3ch = pan_img[:3]
        
    hr_grid = make_grid([rgb_img[:3], pan_3ch[:3], ps_img[:3]], normalize=True, scale_each=True, padding=8)
    showtensor(hr_grid, figsize=14, title=f"High-Resolution: RGB (Left) | Panchromatic (Center) | Pansharpened (Right) - {aoi_name}")
    
    # 3. All 8 Low-Resolution Temporal Revisits
    lr_grid = make_grid(lr[:, :3], normalize=True, scale_each=True, nrow=4, padding=4)
    showtensor(lr_grid, figsize=12, title=f"Sentinel-2 Low-Resolution Revisits (All {len(lr)} time-steps)")
    
    # 4. Cloud Masks (CLM)
    lrc_grid = make_grid(lrc, normalize=True, scale_each=True, nrow=4, padding=4)
    showtensor(lrc_grid, figsize=12, title=f"Sentinel-2 Cloud Masks (CLM)")

_ = show_aoi(
    view_image, 
    i=widgets.IntSlider(min=0, max=len(dataset) - 1, step=1, value=min(1467, len(dataset) - 1), layout=Layout(width='95%'))
)


In [ ]:
# Cell 36: Interactive Geospatial Mapping with Folium & Satellite Overlay
import os
import numpy as np
import rasterio as rio
import folium
from glob import glob
from pathlib import Path
from folium.plugins import MarkerCluster
from shapely.geometry.polygon import Polygon

# Prepare metadata lookup (without overwriting the 'dataset' DictDataset object!)
metadata_lookup = df.copy()
if 'index' in metadata_lookup.columns:
    metadata_lookup.set_index('index', inplace=True)

# Discover available AOIs matching hr_dataset
aois_paths = glob(os.path.join(dataset_root, hr_dataset_folder, '*'))
aois = [Path(p).name for p in aois_paths if os.path.isdir(p)]
aois = sorted(aois)
print(f"Found {len(aois)} AOIs available for geospatial mapping.")

def create_map(centre_lat=0, centre_lon=0, zoom_start=14):
    """Creates a Folium Leaflet map with satellite imagery basemap."""
    m = folium.Map(location=[centre_lat, centre_lon], zoom_start=zoom_start, tiles="OpenStreetMap")
    
    # Add Esri High-Resolution World Satellite Imagery
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery &mdash; Source: Esri, Maxar, Earthstar Geographics",
        name="Esri Satellite",
        overlay=False,
        control=True
    ).add_to(m)
    
    folium.LayerControl().add_to(m)
    return m

def read_image_for_overlay(aoi, dataset_root, hr_dataset_folder):
    """Reads and normalizes the HR satellite image to uint8 for Folium overlay."""
    # Look for pansharpened image first, fallback to rgbn
    candidates = glob(os.path.join(dataset_root, hr_dataset_folder, aoi, "*_ps.tiff"))
    if not candidates:
        candidates = glob(os.path.join(dataset_root, hr_dataset_folder, aoi, "*_rgbn.tiff"))
    if not candidates:
        return None
    
    with rio.open(candidates[0]) as src:
        # Read RGB channels (1, 2, 3)
        image = src.read((1, 2, 3)).astype(np.float32)
    
    # Robust percentile normalization to [0, 255] for clear web rendering
    p2, p98 = np.percentile(image, (2, 98))
    image = np.clip((image - p2) / (p98 - p2 + 1e-6) * 255.0, 0, 255).astype(np.uint8)
    
    # Convert from (C, H, W) to (H, W, C) for Folium
    return image.transpose(1, 2, 0)

def add_aoi_to_map(aoi_name, dataset_root, hr_dataset_folder):
    """Builds map with AOI bounding box, metadata tooltip, and satellite overlay."""
    if aoi_name not in metadata_lookup.index:
        print(f"Metadata not found for AOI: {aoi_name}")
        return create_map()
    
    aoi_meta = metadata_lookup.loc[aoi_name]
    raw_bounds = aoi_meta['bounds']
    bounds = eval(raw_bounds) if isinstance(raw_bounds, str) else raw_bounds
    # bounds is [min_lon, min_lat, max_lon, max_lat]
    min_lon, min_lat, max_lon, max_lat = bounds
    centre_lat = (min_lat + max_lat) / 2.0
    centre_lon = (min_lon + max_lon) / 2.0
    
    # 1. Base map centered on the AOI
    m = create_map(centre_lat=centre_lat, centre_lon=centre_lon, zoom_start=14)
    
    # 2. Add Center Marker with metadata popup
    ipcc = aoi_meta.get('IPCC Class', 'N/A')
    smod = aoi_meta.get('SMOD Class', 'N/A')
    cloud = aoi_meta.get('cloud_cover', 'N/A')
    
    popup_html = f"""
    <b>AOI:</b> {aoi_name}<br>
    <b>Coordinates:</b> ({centre_lat:.4f}, {centre_lon:.4f})<br>
    <b>IPCC Class:</b> {ipcc}<br>
    <b>Urban Class (SMOD):</b> {smod}<br>
    <b>Cloud Cover:</b> {cloud}
    """
    folium.Marker(
        [centre_lat, centre_lon], 
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"AOI: {aoi_name}",
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(m)
    
    # 3. Add AOI Bounding Box Polygon
    folium.Rectangle(
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
        color="#e74c3c",
        weight=2,
        fill=False,
        tooltip=f"{aoi_name} Boundary"
    ).add_to(m)
    
    # 4. Overlay High-Resolution Satellite Image
    img_overlay = read_image_for_overlay(aoi_name, dataset_root, hr_dataset_folder)
    if img_overlay is not None:
        folium.raster_layers.ImageOverlay(
            image=img_overlay,
            bounds=[[min_lat, min_lon], [max_lat, max_lon]],
            opacity=0.85,
            name="High-Res Satellite Tile"
        ).add_to(m)
    
    folium.LatLngPopup().add_to(m)
    return m


In [ ]:
# Cell 37: Interactive Map Slider Widget
show_map_aoi = interact_manual.options(manual_name="Display AOI on Map")

def view_aoi_on_map(i=0):
    aoi_name = aois[i]
    print(f"Displaying [{i}/{len(aois)-1}]: {aoi_name}")
    return add_aoi_to_map(aoi_name, dataset_root, hr_dataset_folder)

_ = show_map_aoi(
    view_aoi_on_map, 
    i=widgets.IntSlider(min=0, max=len(aois) - 1, step=1, value=min(1467, len(aois) - 1), layout=Layout(width='95%'))
)
